# Software Development Trends


## Setup


In [14]:
from io import StringIO

import altair as alt
import attaviz
import pandas as pd
import pycountry
import requests

attaviz.enable()
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [41]:
INNOVATION_GRAPH_DATA_URL = (
    "https://raw.githubusercontent.com/github/innovationgraph/main/data"
)
WORKING_AGE_POPULATION_INDICATOR = "SP.POP.1564.TO"

COUNTRY_NAMES = {
    "BF": "Burkina Faso",
    "CV": "Cabo Verde",
    "GM": "The Gambia",
    "GH": "Ghana",
    "GN": "Guinea",
    "GW": "Guinea-Bissau",
    "CI": "Côte d'Ivoire",
    "LR": "Liberia",
    "ML": "Mali",
    "MR": "Mauritania",
    "NE": "Niger",
    "NG": "Nigeria",
    "SN": "Senegal",
    "SL": "Sierra Leone",
    "TG": "Togo",
    "SA" : "South Africa",
    "IN" : "India"
}
COUNTRY_CODES = list(COUNTRY_NAMES)
COUNTRY_ORDER = list(COUNTRY_NAMES.values())

COUNTRY_COLORS = attaviz.CATEGORICAL[: len(COUNTRY_CODES)]


def country_name_from_iso2(code: str) -> str:
    """Display name for a partner economy, including the EU aggregate."""
    if code == "EU":
        return "European Union"
    if code in COUNTRY_NAMES:
        return COUNTRY_NAMES[code]
    country = pycountry.countries.get(alpha_2=code)
    return country.name if country else code


def add_quarter_start(df: pd.DataFrame) -> pd.DataFrame:
    """Turn the year and quarter columns into a timestamp for the x axis."""
    periods = pd.PeriodIndex.from_fields(
        year=df["year"], quarter=df["quarter"], freq="Q"
    )
    return df.assign(quarter_start=periods.to_timestamp())

## Data

In [42]:
def load_working_age_population() -> pd.DataFrame:
    """Fetch World Bank working-age population (15-64) by country and year."""
    response = requests.get(
        (
            "https://api.worldbank.org/v2/country/"
            f"{';'.join(COUNTRY_CODES)}/indicator/{WORKING_AGE_POPULATION_INDICATOR}"
        ),
        params={"format": "json", "per_page": 20_000},
        timeout=30,
    )
    response.raise_for_status()
    _, observations = response.json()
    return (
        pd.json_normalize(observations)
        .rename(
            columns={
                "country.id": "iso2_code",
                "date": "year",
                "value": "working_age_population",
            }
        )
        .assign(year=lambda d: d["year"].astype(int))
        .dropna(subset=["iso2_code"])
        .loc[lambda d: d["iso2_code"].isin(COUNTRY_CODES)]
        .filter(["iso2_code", "year", "working_age_population"])
        .sort_values(["iso2_code", "year"])
    )


def attach_population(github_df: pd.DataFrame, population: pd.DataFrame):
    """Join population by year, carrying the latest year forward."""
    pieces = []
    for code, group in github_df.groupby("iso2_code", sort=False):
        country_pop = population.loc[population["iso2_code"] == code]
        pieces.append(
            pd.merge_asof(
                group.sort_values("year"),
                country_pop[["year", "working_age_population"]].sort_values("year"),
                on="year",
                direction="backward",
            )
        )
    return pd.concat(pieces, ignore_index=True)


def load_github_metric(metric: str, population: pd.DataFrame) -> pd.DataFrame:
    """Read an Innovation Graph quarterly panel per 100k working-age people."""
    response = requests.get(
        f"{INNOVATION_GRAPH_DATA_URL}/{metric}.csv", timeout=30
    )
    response.raise_for_status()
    raw = (
        pd.read_csv(StringIO(response.text))
        .dropna(subset=["iso2_code"])
        .loc[lambda d: d["iso2_code"].isin(COUNTRY_CODES)]
        .assign(country_name=lambda d: d["iso2_code"].map(COUNTRY_NAMES))
        .pipe(add_quarter_start)
        .sort_values(["iso2_code", "year", "quarter"])
    )
    per_100k = f"{metric}_per_100k"
    return (
        attach_population(raw, population)
        .assign(
            **{per_100k: lambda d: d[metric] / d["working_age_population"] * 100_000}
        )
        .dropna(subset=[per_100k])
        .sort_values(["iso2_code", "year", "quarter"])
    )


population = load_working_age_population()

developers = load_github_metric("developers", population)
repositories = load_github_metric("repositories", population)
organizations = load_github_metric("organizations", population)
git_pushes = load_github_metric("git_pushes", population)

developers.tail()

,developers,iso2_code,year,quarter,country_name,quarter_start,working_age_population,developers_per_100k
420,26970,TG,2025,1,Togo,2025-01-01,4819719,559.576191
421,28565,TG,2025,2,Togo,2025-04-01,4819719,592.669407
422,31619,TG,2025,3,Togo,2025-07-01,4819719,656.034097
419,35980,TG,2025,4,Togo,2025-10-01,4819719,746.516550
423,40421,TG,2026,1,Togo,2026-01-01,4819719,838.658851


## GitHub developers per 100k people over time

In [43]:
def make_trend_chart(
    data: pd.DataFrame,
    raw_col: str,
    value_col: str,
    title: str,
    subtitle: str,
    y_axis_title: str,
    width: int = 620,
    height: int = 340,
):
    """One panel, one line per country, labelled at the last observation.

    The Algeria version drew three panels, one per peer group, because
    sixteen lines do not fit in one. Five fit.
    """
    highlight = alt.selection_point(fields=["country_name"], bind="legend")
    zoom = alt.selection_interval(bind="scales", encodings=["x"])

    # Headroom on the x scale, or Vega clips the end labels at the edge.
    span = data["quarter_start"].max() - data["quarter_start"].min()
    x_domain = [
        data["quarter_start"].min(),
        data["quarter_start"].max() + span * 0.16,
    ]

    color = alt.Color(
        "country_name:N",
        title=None,
        sort=COUNTRY_ORDER,
        scale=alt.Scale(domain=COUNTRY_ORDER, range=COUNTRY_COLORS),
        legend=alt.Legend(orient="bottom", columns=8),
    )

    base = alt.Chart(data).encode(
        x=alt.X("quarter_start:T", title=None, scale=alt.Scale(domain=x_domain)),
        y=alt.Y(
            f"{value_col}:Q",
            title=y_axis_title,
            axis=alt.Axis(format="~s"),
            scale=alt.Scale(zero=True),
        ),
        color=color,
        opacity=alt.when(highlight).then(alt.value(1)).otherwise(alt.value(0.15)),
    )

    lines = (
        base.mark_line(strokeWidth=2.5)
        .encode(
            tooltip=[
                alt.Tooltip("country_name:N", title="Country"),
                alt.Tooltip("year:O", title="Year"),
                alt.Tooltip("quarter:O", title="Quarter"),
                alt.Tooltip(f"{value_col}:Q", title=y_axis_title, format=",.0f"),
                alt.Tooltip(
                    f"{raw_col}:Q",
                    title=raw_col.replace("_", " ").title(),
                    format=",",
                ),
                alt.Tooltip(
                    "working_age_population:Q",
                    title="Working-age population",
                    format=",.0f",
                ),
            ],
        )
        .add_params(highlight, zoom)
    )

    labels = (
        base.transform_window(
            row_number="row_number()",
            sort=[alt.SortField("quarter_start", order="descending")],
            groupby=["country_name"],
        )
        .transform_filter(alt.datum.row_number == 1)
        .mark_text(align="left", baseline="middle", dx=6, fontWeight="bold")
        .encode(text="country_name:N")
    )

    return (lines + labels).properties(
        width=width,
        height=height,
        title=alt.Title(text=title, subtitle=subtitle),
    )


POPULATION_NOTE = (
    "Source: GitHub Innovation Graph; World Bank working-age population (15-64)."
)

developers_chart = make_trend_chart(
    developers,
    raw_col="developers",
    value_col="developers_per_100k",
    title="GitHub developers per 100k working-age people",
    subtitle="Ghana and Nigeria against selected comparators, 2020 Q1 to 2025 Q4",
    y_axis_title="Developers per 100k",
)
developers_chart += (
    alt.Chart(pd.DataFrame({"quarter_start": [pd.Timestamp("2024-10-01")]}))
    .mark_rule(color=attaviz.REFERENCE, strokeDash=[4, 4])
    .encode(x="quarter_start:T")
)
copilot_annotation = pd.DataFrame(
    {
        "label_date": [pd.Timestamp("2024-11-01")],
        "label_height": [developers["developers_per_100k"].max() * 0.89],
        "label": ["GitHub Copilot access\nwas made free"],
    }
)
developers_chart += (
    alt.Chart(copilot_annotation)
    .mark_text(align="left", baseline="middle", dx=6, color=attaviz.REFERENCE)
    .encode(
        x="label_date:T",
        y="label_height:Q",
        text="label:N",
    )
)

attaviz.add_caption(developers_chart, POPULATION_NOTE)

alt.VConcatChart(...)

### Measuring Rate of Change before and After Co-Pilot Access

`quarterly_slope` estimates the **average linear rate of change** in developers per 100k people, per quarter, using ordinary least squares.

**Compute the slope** as the covariance of time and developer count divided
   by the variance of time:

   $$\hat{\beta} = \frac{\sum_i (t_i - \bar{t})(y_i - \bar{y})}{\sum_i (t_i - \bar{t})^2}$$

   where $t_i$ is the quarter index and $y_i$ is `developers_per_100k`.

In [57]:
COPILOT_ACCESS_QUARTER = pd.Timestamp("2024-10-01")


def quarterly_slope(data: pd.DataFrame) -> float:
    """Return the least-squares change in developers per 100k per quarter."""
    quarter_index = data["year"] * 4 + data["quarter"]
    centered_quarters = quarter_index - quarter_index.mean()
    centered_developers = (
        data["developers_per_100k"] - data["developers_per_100k"].mean()
    )
    return (centered_quarters * centered_developers).sum() / (
        centered_quarters**2
    ).sum()


growth_rates = []
for country_name, country_data in developers.groupby("country_name", sort=False):
    periods = {
        "Till 2024 Q4": country_data.loc[
            country_data["quarter_start"] <= COPILOT_ACCESS_QUARTER
        ],
        "From 2025 Q1": country_data.loc[
            country_data["quarter_start"] > COPILOT_ACCESS_QUARTER
        ],
    }
    for period, period_data in periods.items():
        growth_rates.append(
            {
                "country_name": country_name,
                "period": period,
                "slope_per_quarter": quarterly_slope(period_data),
            }
        )

developers_growth_rates = (
    pd.DataFrame(growth_rates)
    .pivot(index="country_name", columns="period", values="slope_per_quarter")
    .reindex(COUNTRY_ORDER)
    .rename_axis(None, axis="columns")
    .rename(columns={
        "Till 2024 Q4": "Till 2024 Q4 (per quarter)",
        "From 2025 Q1": "From 2025 Q1 (per quarter)",
    })
    .round(1)
)

developers_growth_rates.reset_index().sort_values(by="From 2025 Q1 (per quarter)", ascending=False)

,country_name,From 2025 Q1 (per quarter),Till 2024 Q4 (per quarter)
15,South Africa,299.9,84.9
16,India,189.5,72.4
3,Ghana,123.5,40.2
1,Cabo Verde,115.5,62.7
11,Nigeria,89.2,42.0
14,Togo,71.2,20.1
12,Senegal,67.2,18.5
9,Mauritania,56.2,12.6
6,Côte d'Ivoire,55.4,13.4
2,The Gambia,41.3,12.7


**In every West African country without exception, there is a substantial increase in the average per-quarter change in developers per 100k working age population.**

## Repositories, organizations, and git pushes


In [58]:
repositories_chart = make_trend_chart(
    repositories,
    raw_col="repositories",
    value_col="repositories_per_100k",
    title="GitHub repositories per 100k working-age people",
    subtitle="Quarterly public repositories, 2020 Q1 to 2025 Q4",
    y_axis_title="Repositories per 100k",
)

attaviz.add_caption(repositories_chart, POPULATION_NOTE)

alt.VConcatChart(...)

In [66]:
pushes_per_developer = (
    git_pushes.merge(
        developers[
            ["iso2_code", "year", "quarter", "developers_per_100k", "developers"]
        ],
        on=["iso2_code", "year", "quarter"],
        suffixes=("_pushes", "_developers"),
    )
    .assign(
        pushes_per_developer=lambda d: d["git_pushes"] / d["developers"],
        period=lambda d: d["year"].where(
            d["year"] >= 2025, "Till 2024 Q4"
        ).astype(str),
    )
)

country_period_medians = (
    pushes_per_developer.groupby(["country_name", "period"], as_index=False)
    .agg(
        median_developers_per_100k=("developers_per_100k", "median"),
        median_pushes_per_developer=("pushes_per_developer", "median"),
        quarters=("quarter", "count"),
    )
)

scatter_color = alt.Color(
    "country_name:N",
    title=None,
    sort=COUNTRY_ORDER,
    scale=alt.Scale(domain=COUNTRY_ORDER, range=COUNTRY_COLORS),
)

scatter_tooltips = [
    alt.Tooltip("country_name:N", title="Country"),
    alt.Tooltip("period:N", title="Period"),
    alt.Tooltip(
        "median_developers_per_100k:Q",
        title="Median developers per 100k",
        format=".3f",
    ),
    alt.Tooltip(
        "median_pushes_per_developer:Q",
        title="Median pushes per developer",
        format=".3f",
    ),
    alt.Tooltip("quarters:Q", title="Quarters"),
]


def make_pushes_scatter(period: str, show_x_axis: bool, show_legend: bool):
    x_axis = (
        alt.Axis(title="Median developers per 100k working-age people")
        if show_x_axis
        else None
    )
    color = scatter_color if show_legend else alt.Color(
        "country_name:N",
        sort=COUNTRY_ORDER,
        scale=alt.Scale(domain=COUNTRY_ORDER, range=COUNTRY_COLORS),
        legend=None,
    )
    return (
        alt.Chart(country_period_medians)
        .transform_filter(alt.datum.period == period)
        .mark_circle(size=100, opacity=0.85)
        .encode(
            x=alt.X("median_developers_per_100k:Q", axis=x_axis),
            y=alt.Y(
                "median_pushes_per_developer:Q",
                title="Median git pushes per developer",
            ),
            color=color,
            tooltip=scatter_tooltips,
        )
        .properties(width=200, height=340, title=period)
    )


pushes_per_developer_scatter = (
    alt.hconcat(
        make_pushes_scatter("Till 2024 Q4", show_x_axis=False, show_legend=False),
        make_pushes_scatter("2025", show_x_axis=True, show_legend=True),
        make_pushes_scatter("2026", show_x_axis=False, show_legend=False),
        spacing=12,
    )
    .resolve_scale(x="shared", y="shared")
    .properties(
        title=alt.Title(
            text="Median git pushes per developer and developer density",
            subtitle="Country medians through 2024 Q4, in 2025, and in 2026",
        )
    )
)

attaviz.add_caption(pushes_per_developer_scatter, POPULATION_NOTE)

alt.VConcatChart(...)

In [46]:
organizations_chart = make_trend_chart(
    organizations,
    raw_col="organizations",
    value_col="organizations_per_100k",
    title="GitHub organizations per 100k working-age people",
    subtitle="Quarterly organizations, 2020 Q1 to 2025 Q4",
    y_axis_title="Organizations per 100k",
)

attaviz.add_caption(organizations_chart, POPULATION_NOTE)

alt.VConcatChart(...)

In [47]:
git_pushes_chart = make_trend_chart(
    git_pushes,
    raw_col="git_pushes",
    value_col="git_pushes_per_100k",
    title="Git pushes per 100k working-age people",
    subtitle="Quarterly git pushes, 2020 Q1 to 2025 Q4",
    y_axis_title="Pushes per 100k",
)

attaviz.add_caption(git_pushes_chart, POPULATION_NOTE)

alt.VConcatChart(...)

## Top GitHub language clusters


In [80]:
LANGUAGE_CLUSTER_MAPPING_URL = (
    "https://raw.githubusercontent.com/sandorjuhasz/eci_software/"
    "main/data/outputs/language_to_cluster_mapping.csv"
)
response = requests.get(LANGUAGE_CLUSTER_MAPPING_URL, timeout=30, verify=False)
response.raise_for_status()
clusters = pd.read_csv(StringIO(response.text))

response = requests.get(f"{INNOVATION_GRAPH_DATA_URL}/languages.csv", timeout=30)
response.raise_for_status()
languages = (
    pd.read_csv(StringIO(response.text))
    .dropna(subset=["iso2_code"])
    .loc[lambda d: d["iso2_code"].isin(COUNTRY_CODES)]
    .assign(
        country_name=lambda d: d["iso2_code"].map(COUNTRY_NAMES),
        cluster=lambda d: d["language"].map(
            clusters.set_index("Language")["Cluster Name"]
        ),
    )
    .pipe(add_quarter_start)
)

latest_quarter = languages["quarter_start"].max()
latest_rows = languages.loc[languages["quarter_start"] == latest_quarter]
latest_language_label = (
    f"{latest_rows['year'].iloc[0]} Q{latest_rows['quarter'].iloc[0]}"
)

unmapped_share = (
    latest_rows.loc[latest_rows["cluster"].isna(), "num_pushers"].sum()
    / latest_rows["num_pushers"].sum()
)
print(f"latest quarter: {latest_language_label}")
print(f"pushers in unmapped languages: {unmapped_share:.1%}")

/Users/ssarva/west-africa-labor-market-analysis/.venv/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


latest quarter: 2026 Q1
pushers in unmapped languages: 2.1%


In [81]:
top_clusters = (
    latest_rows.dropna(subset=["cluster"])
    .groupby(["iso2_code", "country_name", "cluster"], as_index=False)["num_pushers"]
    .sum()
    .sort_values(["country_name", "num_pushers"], ascending=[True, False])
    .groupby("iso2_code", as_index=False, group_keys=False)
    .head(10)
    .assign(
        rank=lambda d: (
            d.groupby("iso2_code")["num_pushers"]
            .rank(method="first", ascending=False)
            .astype(int)
        ),
        num_pushers_label=lambda d: d["num_pushers"].map(lambda v: f"{v:,.0f}"),
    )
)

# Each panel keeps its own x and y scale, so a bar shows the shape of a
# country's own ranking rather than its size against India.
#
# The sort has to be an ``EncodingSortField``. The shorthand
# ``sort="-x"`` is resolved per layer, so once the value labels are
# layered on top of the bars Vega-Lite cannot reconcile the two and
# silently falls back to alphabetical order.
#
# For the same reason the bars carry no colour encoding. An
# ``EncodingSortField`` aggregates over every discrete channel in the
# spec, so adding ``color`` would group by cluster AND country and drop
# the sort again. The facet header already names the country, so the
# colour was decoration; the value order is not.
cluster_y = alt.Y(
    "cluster:N",
    sort=alt.EncodingSortField(field="num_pushers", op="sum", order="descending"),
    title=None,
    axis=alt.Axis(labelLimit=170),
)

cluster_bars = (
    alt.Chart(top_clusters)
    .mark_bar(size=11)
    .encode(
        x=alt.X("num_pushers:Q", title="Pushers", axis=alt.Axis(format="~s")),
        y=cluster_y,
        tooltip=[
            alt.Tooltip("country_name:N", title="Country"),
            alt.Tooltip("cluster:N", title="Cluster"),
            alt.Tooltip("num_pushers:Q", title="Pushers", format=","),
        ],
    )
)
cluster_labels = (
    alt.Chart(top_clusters)
    .mark_text(align="left", baseline="middle", dx=4, fontSize=9)
    .encode(x="num_pushers:Q", y=cluster_y, text="num_pushers_label:N")
)

clusters_chart = (
    (cluster_bars + cluster_labels)
    .properties(width=200, height=150)
    .facet(
        facet=alt.Facet(
            "country_name:N",
            title=None,
            sort=COUNTRY_ORDER,
            header=alt.Header(labelFontWeight=600),
        ),
        columns=2,
    )
    .resolve_scale(x="independent", y="independent")
    .properties(
        title=alt.Title(
            text="Top GitHub language clusters",
            subtitle=("Top 10 clusters by pushers"),
        )
    )
)

attaviz.add_caption(
    clusters_chart,
    "Source: GitHub Innovation Graph languages data.\n"
    f"Languages outside the 142-language cluster mapping ({unmapped_share:.1%} "
    "of pushers) are dropped.",
)

alt.VConcatChart(...)

## Top GitHub collaborators

The economy collaborator data records cross-economy collaboration
weights. For each country the chart combines rows where it is the source
with rows where it is the destination, which is what the reference
notebook does.

The edges are directional and asymmetric, so the two sides are separate
measurements rather than the same relationship counted twice.


In [76]:
response = requests.get(
    f"{INNOVATION_GRAPH_DATA_URL}/economy_collaborators.csv", timeout=30
)
response.raise_for_status()
economy_collaborators = pd.read_csv(StringIO(response.text)).dropna(
    subset=["source", "destination"]
)

economy_collaborators = economy_collaborators[
    #(economy_collaborators['source'] != 'EU') |
      (economy_collaborators['destination'] != 'EU')
]

latest_year = int(economy_collaborators["year"].max())
latest_q = int(
    economy_collaborators.loc[
        economy_collaborators["year"] == latest_year, "quarter"
    ].max()
)
latest_collab = economy_collaborators.loc[
    (economy_collaborators["year"] == latest_year)
    & (economy_collaborators["quarter"] == latest_q)
]
latest_collab_label = f"{latest_year} Q{latest_q}"


def partners_for(code: str) -> pd.DataFrame:
    """Every partner of *code* in the latest quarter, both directions."""
    edges = latest_collab.loc[
        (latest_collab["source"] == code) | (latest_collab["destination"] == code)
    ]
    partner = edges["destination"].where(edges["source"] == code, edges["source"])
    return edges.assign(focal_iso2=code, partner_iso2=partner)


top_collaborators = (
    pd.concat([partners_for(code) for code in COUNTRY_CODES], ignore_index=True)
    .groupby(["focal_iso2", "partner_iso2"], as_index=False)["weight"]
    .sum()
    .assign(
        focal_name=lambda d: d["focal_iso2"].map(COUNTRY_NAMES),
        partner_name=lambda d: d["partner_iso2"].map(country_name_from_iso2),
        weight_label=lambda d: d["weight"].map(lambda v: f"{v:,.0f}"),
    )
    .sort_values(["focal_name", "weight"], ascending=[True, False])
    .groupby("focal_iso2", as_index=False, group_keys=False)
    .head(10)
)

partner_y = alt.Y(
    "partner_name:N",
    sort=alt.EncodingSortField(field="weight", op="sum", order="descending"),
    title=None,
    axis=alt.Axis(labelLimit=130),
)

collab_bars = (
    alt.Chart(top_collaborators)
    .mark_bar(size=12)
    .encode(
        x=alt.X(
            "weight:Q",
            title="Collaboration weight",
            axis=alt.Axis(format="~s"),
        ),
        y=partner_y,
        tooltip=[
            alt.Tooltip("focal_name:N", title="Country"),
            alt.Tooltip("partner_name:N", title="Partner"),
            alt.Tooltip("weight:Q", title="Collaboration weight", format=","),
        ],
    )
)
collab_labels = (
    alt.Chart(top_collaborators)
    .mark_text(align="left", baseline="middle", dx=4, fontSize=9)
    .encode(x="weight:Q", y=partner_y, text="weight_label:N")
)

collaborators_chart = (
    (collab_bars + collab_labels)
    .properties(width=200, height=155)
    .facet(
        facet=alt.Facet(
            "focal_name:N",
            title=None,
            sort=COUNTRY_ORDER,
            header=alt.Header(labelFontWeight=600),
        ),
        columns=2,
    )
    .resolve_scale(x="independent", y="independent")
    .properties(
        title=alt.Title(
            text="Top GitHub collaborators",
            subtitle=("Top 10 partner economies by combined weight"),
        )
    )
)

attaviz.add_caption(
    collaborators_chart, "Source: GitHub Innovation Graph economy collaborators data."
)

alt.VConcatChart(...)